### 1. Setup & Library Imports

In [83]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns; sns.set_style('whitegrid')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import plotly.express as px
plt.rcParams['figure.dpi'] = 150

### 2. Load Dataset & Exploratory Data Analysis 

#### 2.1 Loading original datasets

In [84]:
df_train = pd.read_csv('fraudTrain.csv')
df_test = pd.read_csv('fraudTest.csv')

#### 2.2 Adding datasets flags

In [85]:
df_train['data_type'] = 'train'
df_test['data_type'] = 'test'

#### 2.3 Creating the masters dataset

In [86]:
transactions_master = pd.concat([df_train, df_test], ignore_index=True)
print(f"✅ Master dataset: {transactions_master.shape}")
print(transactions_master.head(3))

✅ Master dataset: (1604294, 24)
   ID trans_date_trans_time        cc_num                         merchant  \
0   0         1/1/2019 0:00  2.703190e+15       fraud_Rippin, Kub and Mann   
1   1         1/1/2019 0:00  6.304230e+11  fraud_Heller, Gutmann and Zieme   
2   2         1/1/2019 0:00  3.885950e+13             fraud_Lind-Buckridge   

        category     amt      first     last gender  \
0       misc_net    4.97   Jennifer    Banks      F   
1    grocery_pos  107.23  Stephanie     Gill      F   
2  entertainment  220.11     Edward  Sanchez      M   

                         street  ...      long city_pop  \
0                561 Perry Cove  ...  -81.1781     3495   
1  43039 Riley Greens Suite 393  ... -118.2105      149   
2      594 White Dale Suite 530  ... -112.2620     4154   

                                 job        dob  \
0          Psychologist, counselling   3/9/1988   
1  Special educational needs teacher  6/21/1978   
2        Nature conservation officer  1/19/1

#### 2.4 Fraud Rate Check

In [87]:
print("\nFraud distribution:")
print(transactions_master.is_fraud.value_counts(normalize=True))
print(f"Train fraud: {df_train.is_fraud.mean():.3%}")
print(f"Test fraud:  {df_test.is_fraud.mean():.3%}")


Fraud distribution:
is_fraud
0    0.994919
1    0.005081
Name: proportion, dtype: float64
Train fraud: 0.573%
Test fraud:  0.386%


#### 2.5 Dataset Imbalance + Dimensions

In [88]:
# Cell 8: Dataset Imbalance + Dimensions  
print("📊 DATASET DIMENSIONS & IMBALANCE")
print("="*50)

for split in ['train', 'test', 'All']:
    subset = transactions_master[transactions_master.data_type == split] if split != 'All' else transactions_master
    
    fraud_rate = subset.is_fraud.mean()
    imbalance_ratio = 1 / fraud_rate  # How many legit txns per fraud
    
    print(f"{split:>6}: {subset.shape[0]:>8,} txns | "
          f"Fraud: {fraud_rate:.3%} | "
          f"Ratio: 1:{imbalance_ratio:.0f}")

print(f"\nCards: {transactions_master.cc_num.nunique():,} | "
      f"Merchants: {transactions_master.merchant.nunique():,}")
print("train/test split:", transactions_master.data_type.value_counts().to_dict())

📊 DATASET DIMENSIONS & IMBALANCE
 train: 1,048,575 txns | Fraud: 0.573% | Ratio: 1:175
  test:  555,719 txns | Fraud: 0.386% | Ratio: 1:259
   All: 1,604,294 txns | Fraud: 0.508% | Ratio: 1:197

Cards: 959 | Merchants: 693
train/test split: {'train': 1048575, 'test': 555719}


#### 2.6 Summary Statistics

In [89]:
print("\nTransaction amounts:")
print(transactions_master.amt.describe())


Transaction amounts:
count    1.604294e+06
mean     6.997209e+01
std      1.588492e+02
min      1.000000e+00
25%      9.640000e+00
50%      4.739000e+01
75%      8.304000e+01
max      2.894890e+04
Name: amt, dtype: float64


#### 2.7 Data Quality

In [90]:
print("\nMissing values:")
print(transactions_master.isnull().sum())


Missing values:
ID                       0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
data_type                0
dtype: int64


### 3. Feature Engineering

#### 3.1 Loading SQL queries

In [91]:
# Load the SQL CSV files
q37 = pd.read_csv('17_Intermediate_Question_Solved.csv')
q38 = pd.read_csv('18_Intermediate_Question_Solved.csv')
q39 = pd.read_csv('19_Intermediate_Question_Solved.csv')
q40 = pd.read_csv('20_Intermediate_Question_Solved.csv')

print("✅ SQL Features loaded:")
print("Q37:", q37.shape, q37.columns.tolist())
print("Q38:", q38.shape, q38.columns.tolist())
print("Q39:", q39.shape, q39.columns.tolist())
print("Q40:", q40.shape, q40.columns.tolist())

✅ SQL Features loaded:
Q37: (959, 4) ['cc_num', 'top_3_merchant_amount', 'total_card_amount', 'top_3_concentration_pct']
Q38: (693, 4) ['merchant', 'top_5_cc_amount', 'total_card_amount', 'top_5_concentration_pct']
Q39: (539, 2) ['cc_num', 'days_total']
Q40: (959, 3) ['cc_num', 'fraud_before', 'fraud_after']


#### 3.2 Time features

In [92]:
transactions_master['trans_datetime'] = pd.to_datetime(transactions_master['trans_date_trans_time'])
transactions_master['hour'] = transactions_master['trans_datetime'].dt.hour
transactions_master['day_of_week'] = transactions_master['trans_datetime'].dt.dayofweek
transactions_master['is_weekend'] = transactions_master['day_of_week'].isin([5, 6])
transactions_master['is_night'] = transactions_master['hour'].isin([0, 1, 2, 3, 22, 23])

print("✅ Time features added")

✅ Time features added


In [93]:
transactions_master.head(5)

,ID,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,unix_time,merch_lat,merch_long,is_fraud,data_type,trans_datetime,hour,day_of_week,is_weekend,is_night
0,0,1/1/2019 0:00,2.703190e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,1325376018,36.011293,-82.048315,0,train,2019-01-01 00:00:00,0,1,False,True
1,1,1/1/2019 0:00,6.304230e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,1325376044,49.159047,-118.186462,0,train,2019-01-01 00:00:00,0,1,False,True
2,2,1/1/2019 0:00,3.885950e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,1325376051,43.150704,-112.154481,0,train,2019-01-01 00:00:00,0,1,False,True
3,3,1/1/2019 0:01,3.534090e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,1325376076,47.034331,-112.561071,0,train,2019-01-01 00:01:00,0,1,False,True
4,4,1/1/2019 0:03,3.755340e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,1325376186,38.674999,-78.632459,0,train,2019-01-01 00:03:00,0,1,False,True


#### 3.3 Prepare SQL Features

In [94]:
# Q37: keep as is
q37 = q37[['cc_num', 'top_3_concentration_pct']]

# Q38: keep as is
q38 = q38[['merchant', 'top_5_concentration_pct']]

# Q39: rename days_total to high_velocity_days
q39 = q39[['cc_num', 'days_total']].rename(columns={'days_total': 'high_velocity_days'})

# Q40: create fraud delta from before/after
q40 = q40[['cc_num', 'fraud_before', 'fraud_after']].copy()
q40['fraud_delta_pct'] = q40['fraud_after'] - q40['fraud_before']

print("✅ Prepared feature tables:")
print("Q37:", q37.shape)
print("Q38:", q38.shape)
print("Q39:", q39.shape)
print("Q40:", q40.shape)

✅ Prepared feature tables:
Q37: (959, 2)
Q38: (693, 2)
Q39: (539, 2)
Q40: (959, 4)


#### 3.4 Merge SQL features into transactions_master

In [95]:
transactions_master = transactions_master.merge(q37, on='cc_num', how='left')
transactions_master = transactions_master.merge(q38, on='merchant', how='left')
transactions_master = transactions_master.merge(q39, on='cc_num', how='left')
transactions_master = transactions_master.merge(q40[['cc_num', 'fraud_delta_pct']], on='cc_num', how='left')

print("✅ SQL features merged")
print(transactions_master[
    ['top_3_concentration_pct', 'top_5_concentration_pct', 'high_velocity_days', 'fraud_delta_pct']
].describe())

✅ SQL features merged
       top_3_concentration_pct  top_5_concentration_pct  high_velocity_days  \
count             1.604294e+06             1.604294e+06        1.284635e+06   
mean              6.043273e+00             5.096301e+00        2.683186e+01   
std               3.748060e+00             4.920411e+00        3.106677e+01   
min               2.180000e+00             2.140000e+00        1.000000e+00   
25%               3.650000e+00             2.830000e+00        5.000000e+00   
50%               5.000000e+00             3.360000e+00        1.500000e+01   
75%               7.190000e+00             6.350000e+00        3.800000e+01   
max               8.974000e+01             4.588000e+01        2.490000e+02   

       fraud_delta_pct  
count     1.603570e+06  
mean     -1.190637e-01  
std       1.198701e+00  
min      -4.120000e+00  
25%      -8.000000e-01  
50%      -3.200000e-01  
75%       6.200000e-01  
max       5.780000e+00  


In [97]:
transactions_master.head(10)

,ID,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,data_type,trans_datetime,hour,day_of_week,is_weekend,is_night,top_3_concentration_pct,top_5_concentration_pct,high_velocity_days,fraud_delta_pct
0,0,1/1/2019 0:00,2.703190e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,train,2019-01-01 00:00:00,0,1,False,True,4.17,6.99,17.0,1.02
1,1,1/1/2019 0:00,6.304230e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,train,2019-01-01 00:00:00,0,1,False,True,6.12,3.54,62.0,0.44
2,2,1/1/2019 0:00,3.885950e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,train,2019-01-01 00:00:00,0,1,False,True,9.23,3.58,NaN,2.90
3,3,1/1/2019 0:01,3.534090e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,train,2019-01-01 00:01:00,0,1,False,True,19.37,2.22,NaN,-4.12
4,4,1/1/2019 0:03,3.755340e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,train,2019-01-01 00:03:00,0,1,False,True,2.98,6.02,9.0,1.20
5,5,1/1/2019 0:04,4.767270e+15,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,...,train,2019-01-01 00:04:00,0,1,False,True,16.17,2.39,NaN,2.95
6,6,1/1/2019 0:04,3.007470e+13,fraud_Rowe-Vandervort,grocery_net,44.54,Kelsey,Richards,F,889 Sarah Station Suite 624,...,train,2019-01-01 00:04:00,0,1,False,True,2.91,4.21,20.0,-1.07
7,7,1/1/2019 0:05,6.011360e+15,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,...,train,2019-01-01 00:05:00,0,1,False,True,4.59,2.37,NaN,-0.95
8,8,1/1/2019 0:05,4.922710e+15,fraud_Herzog Ltd,misc_pos,4.27,Heather,Chase,F,6888 Hicks Stream Suite 954,...,train,2019-01-01 00:05:00,0,1,False,True,4.20,7.14,NaN,-1.67
9,9,1/1/2019 0:06,2.720830e+15,"fraud_Schoen, Kuphal and Nitzsche",grocery_pos,198.39,Melissa,Aguilar,F,21326 Taylor Squares Suite 708,...,train,2019-01-01 00:06:00,0,1,False,True,4.12,3.28,NaN,-1.35


#### 3.5 Final feature list

In [98]:
feature_cols = [
    'amt',
    'top_3_concentration_pct',
    'top_5_concentration_pct',
    'high_velocity_days',
    'fraud_delta_pct',
    'hour',
    'is_weekend',
    'is_night'
]

print("🎯 ML features ready:", feature_cols)
print(transactions_master[feature_cols].head())

transactions_master.to_csv('transactions_master_features.csv', index=False)
print("💾 Saved: transactions_master_features.csv")

🎯 ML features ready: ['amt', 'top_3_concentration_pct', 'top_5_concentration_pct', 'high_velocity_days', 'fraud_delta_pct', 'hour', 'is_weekend', 'is_night']
      amt  top_3_concentration_pct  top_5_concentration_pct  \
0    4.97                     4.17                     6.99   
1  107.23                     6.12                     3.54   
2  220.11                     9.23                     3.58   
3   45.00                    19.37                     2.22   
4   41.96                     2.98                     6.02   

   high_velocity_days  fraud_delta_pct  hour  is_weekend  is_night  
0                17.0             1.02     0       False      True  
1                62.0             0.44     0       False      True  
2                 NaN             2.90     0       False      True  
3                 NaN            -4.12     0       False      True  
4                 9.0             1.20     0       False      True  
💾 Saved: transactions_master_features.csv
